
| Buổi | Chủ đề | Kỹ năng |
|:---:|--------|:-------:|
| Buổi 1 | Project Introduction & Setup | 🟢 |
| Buổi 2 | Business Questions & Dataset Design | 🟢 |
| Buổi 3 | Data Generation & Loading với Pandas | 🟡 |
| Buổi 4 | Data Cleaning & Validation | 🟡 |
| **📌 Buổi 5** | **KPI & Cohort Retention Analysis** | **🔵** |
| Buổi 6 | Portfolio Reporting & Visualization | 🔴 |
| Buổi 7 | Quiz & Submission Checklist | 🔴 |



<div align="center">

# 📊 Buổi 5 — KPI & Cohort Retention Analysis

# Cohort & Retention Analytics Portfolio

*monthly KPI · channel contribution · first purchase cohort · retention matrix*

</div>



## ⚠️ Phần 0 — Chuẩn Bị Môi Trường

> Buổi 5 **load dữ liệu từ `data/processed/`** — khác với Buổi 3/4. Đảm bảo đã chạy toàn bộ Buổi 4 để 3 file sau tồn tại:
> - `data/processed/clean_orders.csv`
> - `data/raw/customers.csv` (cho acquisition_channel)

**Kích hoạt môi trường ảo (nếu chưa):**

```powershell
cd C:\Users\ADMIN\cohort-retention-analytics
venv\Scripts\activate
```

---

### ⚡ Step 1 — Setup imports và path variables


In [1]:

# ═══════════════════════════════════════════════════════════════════════════
# Phần 0: Setup & Imports  (giống Section 2 project_starter.ipynb)
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.0f}".format)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_DIR      = Path.cwd().parent / "cohort-retention-analytics"
RAW_DIR          = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR    = PROJECT_DIR / "data" / "processed"
OUTPUT_TABLE_DIR = PROJECT_DIR / "outputs" / "tables"
OUTPUT_CHART_DIR = PROJECT_DIR / "outputs" / "charts"

print("✅ Setup complete")
print(f"   PROCESSED_DIR : {PROCESSED_DIR}")
print(f"   exists        : {PROCESSED_DIR.exists()}")


✅ Setup complete
   PROCESSED_DIR : c:\Users\ADMIN\cohort-retention-analytics\data\processed
   exists        : True



### ⚡ Step 2 — Load `clean_orders` từ data/processed/

**Tại sao load từ processed/ thay vì raw/?**

| | `orders.csv` (raw) | `clean_orders.csv` (processed) |
|---|:---:|:---:|
| Cột `order_month`, `signup_month` | ❌ | ✅ |
| Đã loại duplicate `order_id` | ❌ | ✅ |
| `analysis_revenue = 0` với cancelled/returned | ❌ | ✅ |
| Đã merge `segment` từ customers | ❌ | ✅ |

> ⚠️ **Lưu ý:** Buổi 4 chỉ merge `signup_date` + `segment` từ customers. Buổi 5 cần thêm `acquisition_channel` cho phân tích kênh → cell dưới sẽ tự động merge nếu thiếu.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Phần 0 (tiếp): Load clean_orders + bổ sung acquisition_channel
# ═══════════════════════════════════════════════════════════════════════════

clean_orders = pd.read_csv(
    PROCESSED_DIR / "clean_orders.csv",
    parse_dates=["order_date", "signup_date", "order_month", "signup_month"],
)

# acquisition_channel cần cho channel_contribution ở Phần 2
# Nếu chưa có trong clean_orders → merge thêm từ customers.csv
if "acquisition_channel" not in clean_orders.columns:
    customers = pd.read_csv(RAW_DIR / "customers.csv")
    clean_orders = clean_orders.merge(
        customers[["customer_id", "acquisition_channel"]],
        on="customer_id",
        how="left",
    )
    print("ℹ️  Đã thêm cột acquisition_channel từ customers.csv")

print(f"✅ clean_orders: {clean_orders.shape[0]:,} rows × {clean_orders.shape[1]} cols")
print(f"\nCác cột: {list(clean_orders.columns)}")
print(f"\nPhân phối status:")
print(clean_orders["status"].value_counts().to_string())


**Kết quả mẫu**

![img5-1](images/images_buoi5/img5-1.png)


---

## 📈 Phần 1 — Monthly KPI Analysis

### KPI là gì và tại sao phải tính theo tháng?

**KPI (Key Performance Indicator)** là các con số đo lường hiệu quả kinh doanh. Trong e-commerce, leadership thường hỏi:

> *"Tháng này tốt hơn hay xấu hơn tháng trước?"*

Để trả lời, cần ít nhất 5 KPI cốt lõi:

| KPI | Công thức | Ý nghĩa |
|-----|-----------|---------|
| **Revenue** | `SUM(analysis_revenue)` | Tổng doanh thu từ đơn completed |
| **Orders** | `NUNIQUE(order_id)` | Số đơn hàng thực sự được giao thành công |
| **Active Customers** | `NUNIQUE(customer_id)` | Số khách đặt hàng ít nhất 1 lần trong tháng |
| **AOV** | `Revenue ÷ Orders` | Giá trị trung bình 1 đơn hàng |
| **Revenue Growth** | `(Tháng_hiện_tại - Tháng_trước) ÷ Tháng_trước` | Tốc độ tăng trưởng tháng-over-tháng |
###
> 💡 **Tại sao dùng `analysis_revenue` thay vì `net_revenue`?**
> `analysis_revenue = 0` với cancelled/returned → loại hoàn toàn đơn không đóng góp doanh thu thực tế. Nếu dùng `net_revenue`, bạn sẽ tính cả tiền của đơn bị huỷ → overstate doanh thu.

---

### ⚡ Step 3 — Tính Monthly KPI

> 📌 Copy 2 đoạn code trong Phần 1 và Phần 2 vào **Section 6: KPI Analysis** của `project_starter.ipynb`.


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════
# Phần 1: Monthly KPI
# → Copy vào Section 6: KPI Analysis của project_starter.ipynb
# ═══════════════════════════════════════════════════════════════════════════

monthly_kpi = (
    clean_orders
    .groupby("order_month")
    .agg(
        revenue          =("analysis_revenue", "sum"),
        orders           =("order_id",         "nunique"),
        active_customers =("customer_id",      "nunique"),
    )
    .assign(
        aov                  =lambda t: t["revenue"] / t["orders"],
        revenue_per_customer =lambda t: t["revenue"] / t["active_customers"],
        revenue_growth       =lambda t: t["revenue"].pct_change(),
    )
    .reset_index()
)

display(
    monthly_kpi
    .style.hide(axis="index")
    .format({
        "order_month":          lambda d: d.strftime("%Y-%m"),
        "revenue":              "{:,.0f}",
        "aov":                  "{:,.0f}",
        "revenue_per_customer": "{:,.0f}",
        "revenue_growth":       "{:+.1%}",
    })
)


**Kết quả mẫu**

![img5-2](images/images_buoi5/img5-2.png)


### 🔍 Code này làm gì?

**`.groupby("order_month")`** — gom tất cả đơn hàng cùng tháng lại thành một nhóm. Giống như sắp xếp hoá đơn vào từng hộp dán nhãn "Tháng 1", "Tháng 2"...

**`.agg(...)`** — với mỗi nhóm tháng, tính tổng doanh thu, đếm số đơn, đếm số khách. Tên cột mới đặt luôn trong này cho gọn.

**`.assign(aov=..., revenue_per_customer=...)`** — thêm 2 cột tính toán từ các cột vừa có:
- `aov = revenue / orders` → trung bình 1 đơn hàng trị giá bao nhiêu
- `revenue_per_customer = revenue / active_customers` → mỗi khách mang lại bao nhiêu trong tháng đó

**`.pct_change()`** — so sánh tháng này với tháng trước, kết quả là % tăng/giảm:
```python
# revenue = [100, 120, 90]
# pct_change() → [NaN, +20%, -25%]
#                  ↑ tháng đầu không có tháng trước nên để trống
```

**`{:+.1%}`** — định dạng hiển thị có dấu cộng/trừ rõ ràng: `+20.0%` hoặc `-25.0%`.



---

## 📡 Phần 2 — Channel Contribution

### Tại sao cần phân tích theo kênh?

Doanh thu tổng có thể tăng, nhưng **tăng đến từ kênh nào?** Đây là câu hỏi marketing cần biết để phân bổ ngân sách hiệu quả:

| Câu hỏi | Metric |
|---------|--------|
| Kênh nào mang lại nhiều doanh thu nhất? | `revenue` + `revenue_share` |
| Kênh nào có giá trị đơn hàng cao nhất? | `aov` |
| Kênh nào có nhiều khách nhất? | `customers` |
| Kênh nào hiệu quả nhất (doanh thu / khách)? | `revenue ÷ customers` |

> 💡 **Lưu ý quan trọng:** Phân tích này chỉ dùng đơn **`completed`** — `.loc[clean_orders["is_completed"]]`. Đơn cancelled/returned không nên tính vào revenue share vì chúng không đóng góp doanh thu thực.

---

### ⚡ Step 4 — Channel Contribution

> 📌 Copy code dưới đây vào **Section 6: KPI Analysis** của `project_starter.ipynb` — ngay bên dưới `monthly_kpi`.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Phần 2: Channel Contribution
# → Copy vào Section 6: KPI Analysis của project_starter.ipynb (tiếp)
# ═══════════════════════════════════════════════════════════════════════════

channel_contribution = (
    clean_orders
    .loc[clean_orders["is_completed"]]          # Chỉ lấy đơn completed
    .groupby("acquisition_channel", observed=True)
    .agg(
        revenue  =("analysis_revenue", "sum"),
        orders   =("order_id",         "nunique"),
        customers=("customer_id",      "nunique"),
    )
    .assign(
        aov           =lambda t: t["revenue"] / t["orders"],
        revenue_share =lambda t: t["revenue"] / t["revenue"].sum(),
    )
    .sort_values("revenue", ascending=False)
    .reset_index()
)

display(
    channel_contribution
    .style.hide(axis="index")
    .format({
        "revenue":       "{:,.0f}",
        "aov":           "{:,.0f}",
        "revenue_share": "{:.1%}",
    })
)


**Kết quả mẫu**

![img5-3](images/images_buoi5/img5-3.png)


### 🔍 Code này làm gì?

**`.loc[clean_orders["is_completed"]]`** — lọc chỉ giữ lại các đơn đã hoàn thành (`is_completed = True`). Đơn bị huỷ hoặc hoàn trả không tính — tiền chưa về thì chưa phải doanh thu.

**`observed=True`** — nếu không ghi dòng này, pandas đôi khi tự thêm dòng trống cho các kênh không có đơn nào. `observed=True` bảo pandas "chỉ hiện kênh nào thực sự có dữ liệu".

**`revenue_share = revenue / revenue.sum()`** — tính % doanh thu của mỗi kênh so với tổng. Ví dụ kênh A có 300 triệu, tổng 1 tỷ → `300/1000 = 30%`.

**`.sort_values("revenue", ascending=False)`** — sắp xếp bảng từ kênh doanh thu cao nhất xuống thấp nhất, để kênh quan trọng nhất hiện trên cùng.



---

## 🔬 Phần 3 — Cohort Analysis: Hiểu Khái Niệm

### Cohort là gì?

**Cohort** = nhóm khách hàng được gom lại theo **một đặc điểm chung xảy ra cùng thời điểm**. Trong phân tích retention, ta dùng:

> **First Purchase Cohort** = nhóm khách hàng có đơn `completed` **đầu tiên** trong cùng một tháng.

**Ví dụ trực quan:**

```
Cohort Tháng 1/2025 (100 khách mua lần đầu):
─────────────────────────────────────────────────────────────
  Tháng 1  │ ████████████████████ 100 khách  → Index 0 (100%)
  Tháng 2  │ █████████            45 khách   → Index 1 ( 45%)
  Tháng 3  │ ██████               30 khách   → Index 2 ( 30%)
  Tháng 4  │ ████                 20 khách   → Index 3 ( 20%)
  Tháng 5  │ ███                  15 khách   → Index 4 ( 15%)
─────────────────────────────────────────────────────────────
  → 45% khách quay lại tháng thứ 2 sau lần mua đầu
  → 30% quay lại tháng thứ 3... (retention giảm dần theo thời gian)
```

### Tại sao Cohort Analysis quan trọng hơn chỉ nhìn active users?

| Câu hỏi | Chỉ dùng Active Users | Dùng Cohort Analysis |
|---------|:---:|:---:|
| Tháng này 500 active users — tốt hay xấu? | ❌ Không biết | ✅ So sánh được với cohort trước |
| Khách mới tháng 1 có quay lại không? | ❌ Không thể biết | ✅ Thấy rõ index 0→1→2 |
| Sản phẩm mới tháng 3 có giữ chân khách tốt hơn? | ❌ Không thể so sánh | ✅ Cohort tháng 3 vs tháng 1 |
| Chiến dịch marketing tháng 11 hiệu quả dài hạn? | ❌ Chỉ thấy spike ngắn | ✅ Cohort Nov theo dõi 3–6 tháng |

### Ba khái niệm cốt lõi

| Khái niệm | Định nghĩa | Ví dụ |
|-----------|-----------|-------|
| **first_purchase_month** | Tháng khách hàng có đơn completed đầu tiên | `2025-01-01` |
| **cohort_index** | Số tháng kể từ `first_purchase_month` | 0, 1, 2, 3, ... |
| **retention_rate** | % khách của cohort còn active ở tháng index N | `45%` ở index 1 |
| **cohort_size** | Tổng khách unique trong cohort (= active tại index 0) | 100 khách |



---

### ⚡ Step 5 — Xác định First Purchase Cohort

> 📌 Copy 4 đoạn code trong Phần 3–6 (cohort) vào **Section 7: Cohort Retention Analysis** của `project_starter.ipynb`.

**Logic 3 bước:**

```
Bước 1: Lọc chỉ đơn completed         →  completed_orders
Bước 2: Tìm tháng đầu tiên mỗi khách  →  first_purchase (groupby customer → min month)
Bước 3: Gắn first_purchase_month       →  orders_cohort (merge nhiều→một)
```


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════
# Phần 3: First Purchase Cohort
# → Copy vào Section 7: Cohort Retention Analysis của project_starter.ipynb
# ═══════════════════════════════════════════════════════════════════════════

# Bước 1: Chỉ phân tích cohort trên đơn completed
# (cancelled/returned không phản ánh hành vi mua hàng thực sự)
completed_orders = clean_orders.loc[clean_orders["is_completed"]].copy()

# Bước 2: Tháng mua hàng đầu tiên của mỗi khách hàng
# groupby customer_id → lấy order_month nhỏ nhất (= sớm nhất)
first_purchase = (
    completed_orders
    .groupby("customer_id")
    .agg(first_purchase_month=("order_month", "min"))
    .reset_index()
)

# Bước 3: Gắn first_purchase_month vào mỗi dòng đơn hàng
# validate="many_to_one": nhiều đơn → 1 khách (1 khách có 1 first_purchase_month duy nhất)
orders_cohort = completed_orders.merge(
    first_purchase,
    on="customer_id",
    how="left",
    validate="many_to_one",
)

print(f"✅ orders_cohort: {orders_cohort.shape[0]:,} rows × {orders_cohort.shape[1]} cols")
print(f"\nSample — xem first_purchase_month đã được gắn đúng:")
display(
    orders_cohort[["customer_id", "order_month", "first_purchase_month"]]
    .drop_duplicates(subset=["customer_id"])
    .head(5)
)


**Kết quả mẫu**

![img5-4](images/images_buoi5/img5-4.png)


---

## 🔢 Phần 4 — Cohort Index

### Cohort Index là gì?

**Cohort Index** = số tháng đã trôi qua kể từ tháng mua hàng đầu tiên:

```
Khách A: first_purchase_month = 2025-01
  Đặt hàng tháng 1 → cohort_index = 0  (tháng đầu tiên)
  Đặt hàng tháng 2 → cohort_index = 1  (1 tháng sau)
  Đặt hàng tháng 4 → cohort_index = 3  (3 tháng sau)
  Bỏ không mua     → cohort_index N/A  (không xuất hiện trong data)

Khách B: first_purchase_month = 2025-03
  Đặt hàng tháng 3 → cohort_index = 0
  Đặt hàng tháng 5 → cohort_index = 2
```

**Công thức tính:**
$$\text{cohort\_index} = (\text{order\_year} - \text{cohort\_year}) \times 12 + (\text{order\_month} - \text{cohort\_month})$$

### ⚡ Step 6 — Tính Cohort Index

> 📌 Copy code dưới đây vào **Section 7** của `project_starter.ipynb` — ngay sau `orders_cohort`.


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════
# Phần 4: Cohort Index
# → Copy vào Section 7 của project_starter.ipynb (tiếp)
# ═══════════════════════════════════════════════════════════════════════════

orders_cohort = orders_cohort.assign(
    cohort_index=(
        (orders_cohort["order_month"].dt.year  - orders_cohort["first_purchase_month"].dt.year)  * 12
      + (orders_cohort["order_month"].dt.month - orders_cohort["first_purchase_month"].dt.month)
    )
)

print("✅ cohort_index tính xong")
print(f"\nPhân phối cohort_index (số tháng sau lần mua đầu):")
print(orders_cohort["cohort_index"].value_counts().sort_index().to_string())

print(f"\nSanity check — cohort_index tại tháng mua đầu phải = 0:")
check = orders_cohort.loc[
    orders_cohort["order_month"] == orders_cohort["first_purchase_month"],
    "cohort_index"
].eq(0).all()
print(f"   order_month == first_purchase_month → cohort_index = 0: {'✅' if check else '❌'}")


**Kết quả mẫu**

![img5-5](images/images_buoi5/img5-5.png)


### 🔍 Code này làm gì?

Công thức tính số tháng chênh lệch giữa tháng mua đầu và tháng đặt đơn:

```python
cohort_index = (order_year - cohort_year) * 12 + (order_month - cohort_month)
```

**Tại sao nhân 12?** Để quy đổi năm sang tháng trước khi cộng. Ví dụ:
- Mua lần đầu: tháng 11/2025 — đặt đơn: tháng 2/2026
- `(2026 - 2025) × 12 + (2 - 11) = 12 - 9 =` **3 tháng** ✅

**Tại sao không dùng cách tính ngày rồi chia 30?** Vì các tháng có số ngày khác nhau (28, 30, 31 ngày) → kết quả sẽ bị lẻ, không ra số nguyên đẹp như 1, 2, 3...

**`orders_cohort = orders_cohort.assign(...)`** — thêm cột `cohort_index` vào bảng. Dòng này gán kết quả trở lại vào chính biến `orders_cohort` để lần sau dùng bảng này đã có sẵn cột mới.



---

## 📐 Phần 5 — Retention Matrix

### Retention Matrix là gì?

Retention matrix là bảng 2 chiều tóm tắt toàn bộ hành vi quay lại của khách hàng:

```
                  cohort_index →   0       1       2       3       4  ...
first_purchase_month ↓
2025-01                          100%    45%     30%     20%     15%
2025-02                          100%    48%     32%     22%      ?
2025-03                          100%    42%     28%      ?       ?
2025-04                          100%    50%      ?       ?       ?
2025-05                          100%     ?       ?       ?       ?
                                  ↑               ↑
                           luôn = 100%      tháng gần đây → NaN (chưa có dữ liệu)
```

**Cách đọc:**
- **Hàng (row)**: 1 cohort = nhóm khách cùng tháng mua đầu tiên
- **Cột (column)**: cohort_index = tháng thứ N sau khi mua lần đầu
- **Giá trị**: % khách của cohort đó còn active ở tháng N
- **Index 0 luôn = 100%**: vì đây là chính tháng mua đầu — mọi khách đều "active" ở tháng này

### ⚡ Step 7 — Tính Retention Matrix (3 bước)

> 📌 Copy code dưới đây vào **Section 7** của `project_starter.ipynb` — ngay sau `cohort_index`.


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════
# Phần 5: Retention Matrix
# → Copy vào Section 7 của project_starter.ipynb (tiếp)
# ═══════════════════════════════════════════════════════════════════════════

# ── Bước 1: Đếm unique customers mỗi (cohort, index) ─────────────────────
cohort_counts = (
    orders_cohort
    .groupby(["first_purchase_month", "cohort_index"])
    .agg(active_customers=("customer_id", "nunique"))
    .reset_index()
)

# ── Bước 2: Lấy cohort_size = số khách tại index 0 ───────────────────────
# cohort_size dùng để tính % sau này
cohort_size = (
    cohort_counts
    .loc[cohort_counts["cohort_index"].eq(0),
         ["first_purchase_month", "active_customers"]]
    .rename(columns={"active_customers": "cohort_size"})
)

# ── Bước 3: Tính retention_rate và pivot thành matrix ────────────────────
retention = (
    cohort_counts
    .merge(cohort_size, on="first_purchase_month", how="left")
    .assign(retention_rate=lambda t: t["active_customers"] / t["cohort_size"])
)

retention_matrix = retention.pivot_table(
    index="first_purchase_month",
    columns="cohort_index",
    values="retention_rate",
)

# Format hiển thị — mỗi ô là % với 1 chữ số thập phân
display(
    retention_matrix
    .style
    .format("{:.0%}", na_rep="—")
    .background_gradient(cmap="YlGn", axis=None, low=0.1)
)


**Kết quả mẫu**

![img5-6](images/images_buoi5/img5-6.png)


### 🔍 Code này làm gì?

**Bước 1 — `cohort_counts`:** Đếm xem mỗi cohort có bao nhiêu khách đặt đơn trong từng tháng. Nếu 1 khách đặt 3 đơn trong tháng, vẫn chỉ tính là 1 khách (không tính số đơn, tính số người).

**Bước 2 — `cohort_size`:** Lọc lấy cột `cohort_index = 0` — đây là tháng mua đầu tiên, cũng là lúc cohort đông nhất. Con số này dùng làm "mẫu số" để tính % ở bước 3.

**Bước 3 — `retention_rate = active_customers / cohort_size`:** Chia số khách quay lại cho số khách ban đầu. Ví dụ cohort tháng 1 có 80 khách, tháng 2 có 36 khách quay lại → `36/80 = 45%`.

**`pivot_table`** — gom lại thành bảng dễ nhìn. Trước đó dữ liệu là dạng danh sách dài (mỗi dòng 1 cặp tháng-index), `pivot_table` xếp thành bảng 2 chiều:
```
Trước:                         Sau:
Tháng 1 | index 0 | 100%      cohort_index →  0     1     2
Tháng 1 | index 1 |  45%  →   Tháng 1        100%  45%  30%
Tháng 1 | index 2 |  30%      Tháng 2        100%  48%  32%
```

**Tô màu xanh (`YlGn`)** — ô nào % cao thì xanh đậm, ô nào thấp thì vàng nhạt. Nhìn màu là thấy ngay cohort nào giữ chân khách tốt mà không cần đọc từng con số.

---

> ⚠️ **Đừng lo khi thấy nhiều ô `—` ở góc dưới bên phải.** Những ô đó là các cohort mới (tháng 10, 11...) chưa có đủ thời gian để quan sát → hoàn toàn bình thường. Khi so sánh giữa các cohort, hãy nhìn vào những cohort đã có **ít nhất 3 tháng dữ liệu** trở lên.



---

## 💰 Phần 6 — Revenue Cohort Matrix

### Retention theo khách hàng vs Retention theo doanh thu

| | Customer Retention Matrix | Revenue Cohort Matrix |
|---|---|---|
| **Đo lường** | % khách quay lại | Tổng doanh thu theo cohort + tháng |
| **Câu hỏi trả lời** | Bao nhiêu % khách còn gắn bó? | Cohort nào đóng góp doanh thu lớn nhất? |
| **Giá trị** | 0%–100% | VNĐ tuyệt đối |
| **Khi nào dùng?** | Đánh giá loyalty | Đánh giá LTV (lifetime value) |

> 💡 **Hai matrix bổ sung cho nhau:** Cohort tháng 11 có thể có retention rate cao (nhiều khách quay lại) nhưng revenue thấp (vì discount 50% dịp Black Friday). Revenue cohort giúp phát hiện điều này.

### ⚡ Step 8 — Revenue Cohort Matrix

> 📌 Copy code dưới đây vào **Section 7** của `project_starter.ipynb` — ngay sau `retention_matrix`.


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════
# Phần 6: Revenue Cohort Matrix
# → Copy vào Section 7 của project_starter.ipynb (tiếp)
# ═══════════════════════════════════════════════════════════════════════════

cohort_revenue = (
    orders_cohort
    .groupby(["first_purchase_month", "cohort_index"])
    .agg(revenue=("analysis_revenue", "sum"))
    .reset_index()
)

cohort_revenue_matrix = cohort_revenue.pivot_table(
    index="first_purchase_month",
    columns="cohort_index",
    values="revenue",
    fill_value=0,           # Tháng không có giao dịch → 0 thay vì NaN
)

display(
    cohort_revenue_matrix
    .style
    .format("{:,.0f}", na_rep="—")
    .background_gradient(cmap="Blues", axis=None)
)


**Kết quả mẫu**

![img5-7](images/images_buoi5/img5-7.png)


---

## 💾 Phần 7 — Export Outputs

> 📌 Copy code dưới đây vào **Section 10: Export Outputs** của `project_starter.ipynb`.

Export 4 file phân tích — đây là **deliverables** của Buổi 5, cũng là input cho Buổi 6 (visualization):

| File | Nội dung | Dùng ở Buổi 6 |
|------|----------|---------------|
| `monthly_kpi.csv` | Revenue, orders, active customers, AOV theo tháng | Revenue trend chart |
| `channel_contribution.csv` | Revenue share, AOV theo channel | Channel bar chart |
| `cohort_retention_matrix.csv` | % retention theo cohort × index | Retention heatmap |
| `cohort_revenue_matrix.csv` | Revenue theo cohort × index | Revenue heatmap |


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Phần 7: Export Outputs
# → Copy vào Section 10: Export Outputs của project_starter.ipynb
# ═══════════════════════════════════════════════════════════════════════════

monthly_kpi.to_csv(
    OUTPUT_TABLE_DIR / "monthly_kpi.csv",
    index=False,
    encoding="utf-8-sig",
)

channel_contribution.to_csv(
    OUTPUT_TABLE_DIR / "channel_contribution.csv",
    index=False,
    encoding="utf-8-sig",
)

# retention_matrix và cohort_revenue_matrix có index quan trọng (first_purchase_month)
# → dùng index=True để giữ lại khi đọc lại
retention_matrix.to_csv(
    OUTPUT_TABLE_DIR / "cohort_retention_matrix.csv",
    encoding="utf-8-sig",
)

cohort_revenue_matrix.to_csv(
    OUTPUT_TABLE_DIR / "cohort_revenue_matrix.csv",
    encoding="utf-8-sig",
)

print("✅ Exported 4 files:")
for f in [
    OUTPUT_TABLE_DIR / "monthly_kpi.csv",
    OUTPUT_TABLE_DIR / "channel_contribution.csv",
    OUTPUT_TABLE_DIR / "cohort_retention_matrix.csv",
    OUTPUT_TABLE_DIR / "cohort_revenue_matrix.csv",
]:
    size_kb = f.stat().st_size / 1024
    print(f"   {'✅' if f.exists() else '❌'} {f.name:<35} {size_kb:.1f} KB")


**Kết quả mẫu**

![img5-8](images/images_buoi5/img5-8.png)


---

## 🔬 Phần 8 — Sanity Checks

Trước khi kết thúc Buổi 5, xác nhận kết quả đúng logic:

```python
# ✅ Check 1: revenue_growth tháng đầu phải là NaN
assert monthly_kpi["revenue_growth"].iloc[0] != monthly_kpi["revenue_growth"].iloc[0], "Tháng đầu phải là NaN"

# ✅ Check 2: Tổng revenue_share = 100%
assert abs(channel_contribution["revenue_share"].sum() - 1.0) < 0.001, "revenue_share phải tổng = 1"

# ✅ Check 3: Cột 0 của retention_matrix phải toàn = 1.0
assert (retention_matrix[0] == 1.0).all(), "cohort_index=0 phải có retention_rate=100%"

# ✅ Check 4: cohort_size của mỗi cohort = nunique customer_id tại index 0
cohort_size_check = orders_cohort.loc[orders_cohort["cohort_index"].eq(0)].groupby("first_purchase_month")["customer_id"].nunique()
print("Cohort sizes:")
print(cohort_size_check.to_string())

# ✅ Check 5: Không có retention_rate > 1.0 (không thể có nhiều hơn 100%)
assert retention_matrix.max().max() <= 1.0, "retention_rate không được vượt quá 100%"

print("\n✅ Tất cả sanity checks đã PASS")
```



---

## ✍️ Bài Tập Buổi 5

---

### 📝 Bài tập 1 — Viết Insight Statements

Sau khi chạy `monthly_kpi` và `retention_matrix`, viết **3 insight statements** theo format:

```
Observation: [Metric] thay đổi [number] ở [period/segment].
Implication: Điều này gợi ý [business meaning].
Recommendation: Nên [action] vì [evidence].
```

**Gợi ý chủ đề:** tháng có revenue cao nhất, channel nào dominant, cohort nào có retention tốt nhất.

---

**💡 Gợi ý đáp án — 3 ví dụ mẫu:**

**Insight 1 — Monthly Revenue:**
```
Observation: Revenue tháng 11/2025 cao hơn tháng 10 khoảng +X% (xem revenue_growth).
Implication: Black Friday promotion tạo spike doanh thu rõ rệt, nhưng cần kiểm tra
             xem spike này đến từ volume đơn hay AOV cao hơn.
Recommendation: Nên so sánh AOV tháng 11 vs tháng 10 — nếu AOV giảm nhưng orders
                tăng mạnh → promotion thu hút khách giá rẻ, cần cân nhắc margin.
```

**Insight 2 — Channel Contribution:**
```
Observation: Kênh [TOP_CHANNEL] chiếm X% tổng revenue nhưng chỉ Y% số khách.
Implication: Khách từ kênh này có AOV cao hơn trung bình — đây là kênh chất lượng cao.
Recommendation: Tăng ngân sách marketing cho kênh này vì LTV/khách cao hơn.
```

**Insight 3 — Cohort Retention:**
```
Observation: Cohort tháng 1/2025 có retention rate tháng 2 = X% (xem retention_matrix[1]).
Implication: Sau lần mua đầu, X% khách quay lại tháng tiếp theo — đây là baseline retention.
Recommendation: Nếu retention < 40%, cần chương trình re-engagement (email reminder,
                loyalty discount) trong 30 ngày sau mua đầu.
```

---

### 📝 Bài tập 2 — Mở rộng KPI

Thêm cột `cumulative_revenue` (doanh thu tích luỹ từ tháng 1) vào `monthly_kpi`:

```python
monthly_kpi["cumulative_revenue"] = monthly_kpi["revenue"].cumsum()
```
**Kết quả mẫu**

![img5-9](images/images_buoi5/img5-9.png)

**Câu hỏi:** `cumsum()` làm gì? Tại sao đây là thông tin có ích cho leadership?

**💡 Đáp án:** `cumsum()` = cumulative sum — tính tổng tích luỹ từng bước:
`[100, 120, 90] → [100, 220, 310]`. Leadership dùng để so sánh với revenue target năm — ví dụ "đến tháng 8 đã đạt 65% target cả năm chưa?"

---

### ✅ Checklist Buổi 5

- [ ] `monthly_kpi` có đủ 6 cột: `revenue`, `orders`, `active_customers`, `aov`, `revenue_per_customer`, `revenue_growth`
- [ ] `channel_contribution` có cột `revenue_share` tổng = **100%**
- [ ] `cohort_index = 0` với mọi đơn hàng có `order_month == first_purchase_month`
- [ ] Cột **0** của `retention_matrix` toàn bộ = **100%**
- [ ] Không có giá trị nào trong `retention_matrix` > 1.0
- [ ] 4 file đã export: `monthly_kpi.csv`, `channel_contribution.csv`, `cohort_retention_matrix.csv`, `cohort_revenue_matrix.csv`
- [ ] Code đã copy vào `project_starter.ipynb`: Section 6 (KPI) và Section 7 (Cohort)
- [ ] Đã viết ít nhất 3 insight statements có số liệu cụ thể
